In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

In [ ]:
# Load dataset - UPDATE THIS PATH to your local file location
df = pd.read_csv("train (1).csv")  # Change to your actual path
print(f"Dataset loaded: {df.shape}")
print(df.head())

In [ ]:
# Fix missing values (avoid inplace warnings)
categorical_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

numeric_cols = ['LoanAmount', 'Loan_Amount_Term']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

print("Missing values after imputation:")
print(df.isnull().sum())

In [ ]:
# ============================================
# IMPORTANT: Create engineered features as NEW COLUMNS in the DataFrame
# These will be the features the model actually uses
# ============================================

# Log transforms (add 1 to avoid log(0))
df['ApplicantIncomeLog'] = np.log(df['ApplicantIncome'] + 1)
df['LoanAmountLog'] = np.log(df['LoanAmount'] + 1)
df['Loan_Amount_Term_Log'] = np.log(df['Loan_Amount_Term'] + 1)

# Total income and its log
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['Total_Income_Log'] = np.log(df['Total_Income'] + 1)

print("Engineered features created: ApplicantIncomeLog, LoanAmountLog, Loan_Amount_Term_Log, Total_Income_Log")
print(df.head())

In [ ]:
# Define what RAW features users will provide (input to pipeline)
raw_categorical_features = [
    'Gender', 'Married', 'Dependents',
    'Education', 'Self_Employed', 'Property_Area'
]

raw_numeric_features = [
    'ApplicantIncome',
    'CoapplicantIncome',
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

# The pipeline will transform numeric features into these FINAL features:
# - One-hot encoded categoricals (dummy variables)
# - The following numeric features (including engineered ones)
final_numeric_features = [
    'ApplicantIncomeLog', 'LoanAmountLog',
    'Loan_Amount_Term_Log', 'Total_Income_Log', 'Credit_History'
]

print("RAW INPUT features (what users provide):")
print("  Categorical:", raw_categorical_features)
print("  Numeric:", raw_numeric_features)
print("\nFINAL numeric features after log transforms:", final_numeric_features)

In [ ]:
# Create ColumnTransformer that does ALL transformations
# Note: It will receive RAW DataFrame and output numeric + one-hot encoded features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), raw_categorical_features),
        ('num', StandardScaler(), final_numeric_features)
    ]
)

In [ ]:
# Prepare training data
# X_train should contain ONLY the raw categorical + raw numeric features
# The engineered features are already in df but will be selected by the ColumnTransformer

# Target encoding
y = df['Loan_Status'].map({'Y': 1, 'N': 0})

# X: ONLY raw features (the engineered ones are already in df but will be accessed by preprocessor)
# We need to include both raw numeric AND categorical in X
X = df[raw_categorical_features + raw_numeric_features]

print("X columns (raw features only):", X.columns.tolist())
print(f"Shape: {X.shape}")
print("\nDistribution of target:")
print(y.value_counts())

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Create full pipeline with XGBoost
from xgboost import XGBClassifier

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', XGBClassifier(
        n_estimators=100,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    ))
])

print("Training pipeline...")
pipeline.fit(X_train, y_train)
print("Training complete!")

In [ ]:
# Evaluate
accuracy = pipeline.score(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

from sklearn.metrics import classification_report
y_pred = pipeline.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Save the complete pipeline
output_file = "loan_model_pipeline.pkl"
joblib.dump(pipeline, output_file)
print(f"Pipeline saved to: {output_file}")

In [1]:
# ============================================
# VERIFY: Test with RAW user inputs
# ============================================

test_raw = pd.DataFrame([{
    'Gender': 'Male',
    'Married': 'No',
    'Dependents': '0',
    'Education': 'Graduate',
    'Self_Employed': 'Yes',
    'Property_Area': 'Urban',
    'ApplicantIncome': 60000,      # Raw
    'CoapplicantIncome': 0,       # Raw
    'LoanAmount': 99,            # Raw
    'Loan_Amount_Term': 16,      # Raw
    'Credit_History': 0.4
}])

print("Test input (RAW - what user provides):")
print(test_raw)

prediction = pipeline.predict(test_raw)
proba = pipeline.predict_proba(test_raw)

print(f"\nPrediction: {prediction[0]}")
print(f"Probabilities: [Reject={proba[0][0]:.4f}, Approve={proba[0][1]:.4f}]")

NameError: name 'pd' is not defined

In [ ]:
# Verify saved pipeline
loaded = joblib.load(output_file)
reload_pred = loaded.predict(test_raw)
print(f"\nReloaded pipeline prediction: {reload_pred[0]}")
print(f"Matches original: {prediction[0] == reload_pred[0]}")
print("[OK] Pipeline saved and loaded correctly!")

In [ ]:
# Check feature schema extraction (what backend will see)
print("\n=== Feature Schema Extraction ===")

col_transformer = pipeline.named_steps['preprocessing']
if hasattr(col_transformer, 'feature_names_in_'):
    raw_features = col_transformer.feature_names_in_.tolist()
    print(f"Raw input features (from ColumnTransformer): {raw_features}")
else:
    print("Warning: No feature_names_in_ - ensure fit on DataFrame")
    raw_features = raw_categorical_features + raw_numeric_features

print("\nExpected for frontend:")
print(f"  Categorical: {raw_categorical_features}")
print(f"  Numeric: {raw_numeric_features}")

# Verify
assert set(raw_features) == set(raw_categorical_features + raw_numeric_features), \
    "Feature names don't match - check ColumnTransformer!"
print("\n[OK] Feature schema is correct!")

## 🎉 Pipeline Training Complete!

The pipeline has been trained and saved as `loan_model_pipeline.pkl`.

### What's in the pipeline:
1. **ColumnTransformer** that:
   - One-hot encodes categorical features (Gender, Education, etc.)
   - Applies StandardScaler to numeric features
   - **Includes engineered features** (logs, totals) automatically
2. **XGBoost classifier** trained on the transformed features

### Key points:
- ✅ Accepts **RAW inputs** (strings like 'Male', raw numbers)
- ✅ No custom classes → **No pickle errors**
- ✅ Feature schema shows **original feature names**
- ✅ Compatible with SHAP/LIME in backend

### Next steps:
1. Upload `loan_model_pipeline.pkl` to your XAI platform
2. Verify the feature schema shows raw features (Gender, Education, ApplicantIncome, etc.)
3. Test prediction from frontend with raw values
4. SHAP and LIME should work correctly

### If you get errors:
- Ensure `train (1).csv` exists at the specified path
- Make sure all required columns are in the CSV
- Check that the ColumnTransformer `feature_names_in_` matches raw features